In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import hashlib
import ast
import re

In [2]:
FILE = '2026-04-19.xlsx' 
path = Path()
file_path = path.absolute() / 'out' / FILE

In [3]:
EXCLUDED_WEIGHTS = [
    '[1, 0, 0, 0, 0]',
    '[0, 1, 0, 0, 0]',
    '[0, 0, 1, 0, 0]',
    '[0, 0, 0, 1, 0]',
    '[0, 0, 0, 0, 1]',
]

In [4]:
df = pd.read_excel(file_path, engine='openpyxl')

In [5]:
df = df[~df["weight"].isin(EXCLUDED_WEIGHTS)].copy()

In [6]:
df = df[~df['instancia'].isna()]

In [7]:
hash_list = df["weight_hash"].unique()

def make_weight_hash_map_from_list(hash_list, start_at=1):
    return {h: rf"$w_{{{i}}}$" for i, h in enumerate(hash_list, start=start_at)}

hash_map = make_weight_hash_map_from_list(hash_list)

In [8]:
df["weight_label"] = df["weight_hash"].map(hash_map).fillna(df["weight_hash"])

In [9]:
hash_map

{'d086fe': '$w_{1}$',
 'edcc2b': '$w_{2}$',
 'acdb7b': '$w_{3}$',
 '6d360f': '$w_{4}$',
 'c264b9': '$w_{5}$',
 '272ddf': '$w_{6}$'}

In [10]:
df['instancia'] = df['instancia'].astype(int)

In [11]:
df['desv_rel_targ_1'] = df['p1'] / df['f1_target']
df['desv_rel_targ_2'] = df['p2'] / df['f2_target']
df['desv_rel_targ_3'] = df['p3'] / df['f3_target']
df['desv_rel_targ_4'] = df['p4'] / df['f4_target']
df['desv_rel_targ_5'] = df['p5'] / df['f5_target']

In [88]:
df['f_sum_t'] = df['f1'] + df['f2'] + df['f3'] + df['f4'] + df['f5']
def _get_ct(df, by):
    df_f_sum_t = pd.pivot_table(df, index=['hash_file', 'weight_label', 'alpha', by], values=['f_sum_t'], aggfunc='sum').reset_index()
    df_f_sum_t_001 = df_f_sum_t.query('alpha == 0.01').drop(columns=['alpha'])
    df_f_sum_t_099 = df_f_sum_t.query('alpha == 0.99').drop(columns=['alpha'])
    df_ct = pd.merge(df_f_sum_t_001, df_f_sum_t_099, on=['hash_file', 'weight_label', by], suffixes=['_001', '_099'])
    df_ct['ct'] = np.abs(df_ct['f_sum_t_099'] - df_ct['f_sum_t_001'])/df_ct['f_sum_t_001']
    return pd.pivot_table(df_ct, index=['weight_label', by], values='ct', aggfunc='mean').reset_index()

def df_to_latex_table(
    df: pd.DataFrame,
    caption=None,
    label=None,
    table_env=True,
    decimal_places=3,
    colname_map=None,
    multirow_first_col=False,
    group_rule=True,
    third_col_title_multicolumn=False,
    weights_in_columns=False,
):
    r"""Gera uma tabela LaTeX (booktabs + table env).

    Se multirow_first_col=True, a primeira coluna (ex.: weights) vira um \multirow agrupando
    linhas consecutivas com o mesmo valor.
    """
    if colname_map is None:
        colname_map = {}

    int_like_cols = {'clientes', 'classe', 'instancia'}

    def _header_label(col):
        name = colname_map.get(col, str(col))
        # Se não parece ser LaTeX, escapa underscores para evitar erro.
        if ('$' not in name) and ('\\' not in name):
            name = name.replace('_', r'\\_')
        return name

    def _is_na(v):
        try:
            return bool(pd.isna(v))
        except Exception:
            return False

    def _fmt(col, val):
        if _is_na(val):
            return ''

        if hasattr(val, 'item'):
            try:
                val = val.item()
            except Exception:
                pass

        if isinstance(val, bool):
            return str(val)
        if isinstance(val, int):
            return str(val)
        if isinstance(val, float):
            if (col in int_like_cols) and val.is_integer():
                return str(int(val))
            if col == 'ct':
                return f'{(val * 100):.1f}\\%'
            return f'{val:.{decimal_places}f}'
    def _sort_weight_cols(cols):
        def _key(c):
            s = str(c)
            m = re.search(r"w_\{(\d+)\}", s)
            return int(m.group(1)) if m else 10**9
        return sorted(list(cols), key=_key)

        return str(val)

    if weights_in_columns:
        if df.empty:
            df_wide = df.copy()
        else:
            if not {'weight_label', 'ct'}.issubset(set(df.columns)):
                raise ValueError("weights_in_columns=True espera colunas 'weight_label' e 'ct'.")

            # identifica a coluna de grupo (clientes ou classe)
            group_cols = [c for c in df.columns if c not in {'weight_label', 'ct'}]
            if len(group_cols) != 1:
                raise ValueError("weights_in_columns=True espera exatamente 1 coluna além de 'weight_label' e 'ct'.")
            group_col = group_cols[0]

            df_wide = pd.pivot_table(
                df,
                index=group_col,
                columns='weight_label',
                values='ct',
                aggfunc='mean',
            )

            # ordena colunas por w_{i}
            ordered_cols = _sort_weight_cols(df_wide.columns)
            df_wide = df_wide.reindex(columns=ordered_cols)

        # construir LaTeX manualmente (2 linhas de header: ct em cima, pesos embaixo)
        num_weights = 0 if df_wide.empty else len(df_wide.columns)
        colspec = 'l' + ('c' * num_weights)

        lines = []
        if table_env:
            lines += [r'\begin{table}[htbp]', r'\centering']
        if caption:
            lines.append(rf'\caption{{{caption}}}')
        if label:
            lines.append(rf'\label{{{label}}}')

        lines.append(rf'\begin{{tabular}}{{{colspec}}}')
        lines.append(r'\toprule')

        ct_title = _header_label('ct')
        if num_weights == 0:
            lines.append(rf'\multicolumn{{1}}{{c}}{{\textbf{{{ct_title}}}}} \\')
            lines.append(r'\midrule')
            lines.append(r'\bottomrule')
            lines.append(r'\end{tabular}')
            if table_env:
                lines.append(r'\end{table}')
            return '\n'.join(lines)

        # linha 1: ct (span todas as colunas)
        lines.append(rf'\multicolumn{{{num_weights + 1}}}{{c}}{{\textbf{{{ct_title}}}}} \\')
        lines.append(r'\midrule')

        # linha 2: pesos
        weight_header = [r'\multicolumn{1}{c}{\textbf{}}'] + [rf'\multicolumn{{1}}{{c}}{{\textbf{{{str(w)}}}}}' for w in df_wide.columns]
        lines.append(' & '.join(weight_header) + r' \\')
        lines.append(r'\midrule')

        # corpo
        for idx_val, row in df_wide.iterrows():
            if 'group_col' in locals() and group_col == 'clientes':
                left = f"{int(idx_val)}C"
            else:
                left = _fmt('group', idx_val)

            out = [left]
            for w in df_wide.columns:
                out.append(_fmt('ct', row[w]))
            lines.append(' & '.join(out) + r' \\')

        lines.append(r'\bottomrule')
        lines.append(r'\end{tabular}')
        if table_env:
            lines.append(r'\end{table}')

        return '\n'.join(lines)

    # Alinhamento: texto à esquerda para não numéricos, centralizado para numéricos
    colspec_parts = []
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            colspec_parts.append('c')
        else:
            colspec_parts.append('l')
    colspec = ''.join(colspec_parts) if colspec_parts else 'l'

    lines = []
    if table_env:
        lines += [r'\begin{table}[htbp]', r'\centering']
    if caption:
        lines.append(rf'\caption{{{caption}}}')
    if label:
        lines.append(rf'\label{{{label}}}')

    lines.append(rf'\begin{{tabular}}{{{colspec}}}')
    lines.append(r'\toprule')


    if third_col_title_multicolumn and len(df.columns) == 3:
        third_title = _header_label(df.columns[2])
        lines.append(rf'\multicolumn{{3}}{{c}}{{\textbf{{{third_title}}}}} \\')
        lines.append(r'\midrule')

        header = [
            rf'\multicolumn{{1}}{{c}}{{\textbf{{{_header_label(df.columns[0])}}}}}',
            rf'\multicolumn{{1}}{{c}}{{\textbf{{{_header_label(df.columns[1])}}}}}',
            r'\multicolumn{1}{c}{\textbf{}}',
        ]
        lines.append(' & '.join(header) + r' \\')
        lines.append(r'\midrule')
    else:
        header = [rf'\multicolumn{{1}}{{c}}{{\textbf{{{_header_label(c)}}}}}' for c in df.columns]
        lines.append(' & '.join(header) + r' \\')
        lines.append(r'\midrule')

    if (not multirow_first_col) or df.empty or len(df.columns) == 0:
        for _, row in df.iterrows():
            out = [_fmt(col, val) for col, val in row.items()]
            lines.append(' & '.join(out) + r' \\')
    else:
        first_col = df.columns[0]
        values = df[first_col].tolist()
        i = 0
        n = len(df)
        while i < n:
            v = values[i]
            j = i + 1
            while j < n and values[j] == v:
                j += 1
            span = j - i

            for k in range(i, j):
                row = df.iloc[k]
                out = []
                for col, val in row.items():
                    if col == first_col:
                        if k == i:
                            out.append(rf'\multirow{{{span}}}{{*}}{{{_fmt(col, val)}}}')
                        else:
                            out.append('')
                    else:
                        out.append(_fmt(col, val))
                lines.append(' & '.join(out) + r' \\')

            if group_rule and j < n:
                lines.append(r'\midrule')
            i = j

    lines.append(r'\bottomrule')
    lines.append(r'\end{tabular}')
    if table_env:
        lines.append(r'\end{table}')

    return '\n'.join(lines)


In [92]:
df_ct_clientes = _get_ct(df, 'clientes')
latex_code = df_to_latex_table(
    df_ct_clientes,
    caption='Sensitivity (ct) by number of clients',
    label='tab:sensitivity_ct_clientes',
    table_env=True,
    decimal_places=3,
    weights_in_columns=True,
    colname_map={'weight_label': 'Weight', 'clientes': 'Clients', 'ct': '$|CT^{(0.99)} - CT^{(0.01)}|/|CT^{(0.01)}|$'},
)
print(latex_code)
df_ct_clientes


\begin{table}[htbp]
\centering
\caption{Sensitivity (ct) by number of clients}
\label{tab:sensitivity_ct_clientes}
\begin{tabular}{lcccccc}
\toprule
\multicolumn{7}{c}{\textbf{$|CT^{(0.99)} - CT^{(0.01)}|/|CT^{(0.01)}|$}} \\
\midrule
\multicolumn{1}{c}{\textbf{}} & \multicolumn{1}{c}{\textbf{$w_{1}$}} & \multicolumn{1}{c}{\textbf{$w_{2}$}} & \multicolumn{1}{c}{\textbf{$w_{3}$}} & \multicolumn{1}{c}{\textbf{$w_{4}$}} & \multicolumn{1}{c}{\textbf{$w_{5}$}} & \multicolumn{1}{c}{\textbf{$w_{6}$}} \\
\midrule
5C & 0.8\% & 0.8\% & 0.3\% & 0.7\% & 0.4\% & 0.7\% \\
10C & 3.3\% & 2.8\% & 2.9\% & 3.4\% & 2.9\% & 3.2\% \\
\bottomrule
\end{tabular}
\end{table}


,weight_label,clientes,ct
0,$w_{1}$,5.0,0.007823
1,$w_{1}$,10.0,0.033183
2,$w_{2}$,5.0,0.007585
3,$w_{2}$,10.0,0.028175
4,$w_{3}$,5.0,0.002655
5,$w_{3}$,10.0,0.028659
6,$w_{4}$,5.0,0.006899
7,$w_{4}$,10.0,0.034412
8,$w_{5}$,5.0,0.004116
9,$w_{5}$,10.0,0.029495


In [91]:
df_ct_classe = _get_ct(df, 'classe')
latex_code = df_to_latex_table(
    df_ct_classe,
    caption='Sensitivity (ct) by class',
    label='tab:sensitivity_ct_classe',
    table_env=True,
    decimal_places=3,
    weights_in_columns=True,
    colname_map={'weight_label': 'Weight', 'clientes': 'Clients', 'ct': '$|CT^{(0.99)} - CT^{(0.01)}|/|CT^{(0.01)}|$'},
)
print(latex_code)
df_ct_classe


\begin{table}[htbp]
\centering
\caption{Sensitivity (ct) by class}
\label{tab:sensitivity_ct_classe}
\begin{tabular}{lcccccc}
\toprule
\multicolumn{7}{c}{\textbf{$|CT^{(0.99)} - CT^{(0.01)}|/|CT^{(0.01)}|$}} \\
\midrule
\multicolumn{1}{c}{\textbf{}} & \multicolumn{1}{c}{\textbf{$w_{1}$}} & \multicolumn{1}{c}{\textbf{$w_{2}$}} & \multicolumn{1}{c}{\textbf{$w_{3}$}} & \multicolumn{1}{c}{\textbf{$w_{4}$}} & \multicolumn{1}{c}{\textbf{$w_{5}$}} & \multicolumn{1}{c}{\textbf{$w_{6}$}} \\
\midrule
1.000 & 1.2\% & 1.5\% & 1.3\% & 1.5\% & 1.3\% & 1.5\% \\
2.000 & 0.2\% & 0.2\% & 0.2\% & 0.2\% & 0.2\% & 0.2\% \\
3.000 & 5.0\% & 3.8\% & 3.5\% & 4.7\% & 4.1\% & 4.6\% \\
4.000 & 1.9\% & 1.8\% & 1.5\% & 2.0\% & 1.4\% & 1.7\% \\
\bottomrule
\end{tabular}
\end{table}


,weight_label,classe,ct
0,$w_{1}$,1.0,0.012424
1,$w_{1}$,2.0,0.002475
2,$w_{1}$,3.0,0.049775
3,$w_{1}$,4.0,0.019251
4,$w_{2}$,1.0,0.014755
5,$w_{2}$,2.0,0.002410
6,$w_{2}$,3.0,0.037623
7,$w_{2}$,4.0,0.018450
8,$w_{3}$,1.0,0.012719
9,$w_{3}$,2.0,0.001708
